In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import glob, time, os
import numpy as np
import pandas as pd

csv_path = glob.glob("/kaggle/input/**/*.csv", recursive=True)[0]
raw = pd.read_csv(csv_path)

print("Loaded:", csv_path)
print("Rows, Columns:", raw.shape)

Loaded: /kaggle/input/datasets/anjalibhegam/multimodal-sports-injury-dataset/multimodal_sports_injury_dataset.csv
Rows, Columns: (15420, 29)


In [2]:
keep_cols = ["athlete_id", "session_id",
             "training_duration", "training_intensity", "training_load",
             "heart_rate", "sleep_quality", "recovery_score", "stress_level",
             "fatigue_index"]

df = raw[keep_cols].copy()
print(df.shape)
df.head()

(15420, 10)


,athlete_id,session_id,training_duration,training_intensity,training_load,heart_rate,sleep_quality,recovery_score,stress_level,fatigue_index
0,1,1,79.106564,5.678475,548.417962,NaN,5.466370,77.899691,0.375825,45.028687
1,1,2,116.954654,NaN,538.043815,40.000000,6.899090,74.845323,0.279165,42.254717
2,1,3,96.354742,6.643242,647.784339,76.174609,9.574152,83.538316,0.334061,61.870056
3,1,4,97.619130,6.194427,336.553431,100.543724,7.614297,79.966618,0.218766,50.268845
4,1,5,83.155527,5.835804,496.076352,78.001235,5.631979,56.945979,0.484086,29.233575


In [3]:
print("=== MISSING VALUES ===")
print(df.isnull().sum())

print("\nFully duplicated rows:", df.duplicated().sum())
print("Repeated athlete+session pairs:", df.duplicated(subset=["athlete_id", "session_id"]).sum())

print("\n=== VALUE RANGES (outliers are NOT deleted) ===")
print(df.describe().T[["min", "mean", "max"]].round(2))

=== MISSING VALUES ===
athlete_id              0
session_id              0
training_duration       0
training_intensity    632
training_load           0
heart_rate            354
sleep_quality         493
recovery_score          0
stress_level            0
fatigue_index           0
dtype: int64

Fully duplicated rows: 0
Repeated athlete+session pairs: 0

=== VALUE RANGES (outliers are NOT deleted) ===
                       min    mean      max
athlete_id            1.00   78.55   156.00
session_id            1.00   49.93   101.00
training_duration    30.00   87.79   180.00
training_intensity    2.00    6.40    10.00
training_load       150.00  670.53  2632.64
heart_rate           40.00   72.69   159.27
sleep_quality         1.16    5.82    10.00
recovery_score        8.90   55.20    98.00
stress_level          0.10    0.42     0.95
fatigue_index        15.00   59.70   270.19


In [4]:
df = df.sort_values(["athlete_id", "session_id"]).reset_index(drop=True)
df["session_number"] = df.groupby("athlete_id").cumcount() + 1

print(df.groupby("athlete_id")["session_number"].max().describe().round(1))

count    156.0
mean      98.8
std        0.9
min       98.0
25%       98.0
50%       99.0
75%       99.0
max      101.0
Name: session_number, dtype: float64


In [5]:
g = df.groupby("athlete_id")

for w in [7, 14]:
    # shift(1) first, so only EARLIER sessions are used, never today's
    df[f"load_sum_{w}"] = g["training_load"].transform(
        lambda s: s.shift(1).rolling(w, min_periods=w).sum())
    df[f"load_mean_{w}"] = g["training_load"].transform(
        lambda s: s.shift(1).rolling(w, min_periods=w).mean())
    df[f"load_minus_mean_{w}"] = df["training_load"] - df[f"load_mean_{w}"]
    df[f"load_ratio_{w}"] = df["training_load"] / df[f"load_mean_{w}"]

for col in ["training_intensity", "heart_rate", "sleep_quality",
            "recovery_score", "stress_level"]:
    for w in [7, 14]:
        df[f"{col}_mean_{w}"] = g[col].transform(
            lambda s: s.shift(1).rolling(w, min_periods=3).mean())

df = df.replace([np.inf, -np.inf], np.nan)
print("Total columns now:", df.shape[1])

Total columns now: 29


In [6]:
a = df[df["athlete_id"] == df["athlete_id"].iloc[0]].reset_index(drop=True)
row15 = a[a["session_number"] == 15].iloc[0]

manual_sum7  = a[(a.session_number >= 8) & (a.session_number <= 14)]["training_load"].sum()
manual_sum14 = a[(a.session_number >= 1) & (a.session_number <= 14)]["training_load"].sum()

assert np.isclose(row15["load_sum_7"], manual_sum7)
assert np.isclose(row15["load_sum_14"], manual_sum14)
print("OK: windows use only earlier sessions.")

OK: windows use only earlier sessions.


In [7]:
model_df = df[df["session_number"] >= 15].copy()          # session 15 = first prediction
train_df = model_df[model_df["session_number"] <= 75].copy()
test_df  = model_df[model_df["session_number"] >= 76].copy()

# Class rule uses TRAINING data only
q1, q2 = train_df["fatigue_index"].quantile([1/3, 2/3])
print("Class cut points:", round(q1, 2), round(q2, 2))

def to_class(x):
    return 0 if x <= q1 else (1 if x <= q2 else 2)

class_names = ["Low", "Moderate", "High"]
train_df["fatigue_class"] = train_df["fatigue_index"].apply(to_class)
test_df["fatigue_class"]  = test_df["fatigue_index"].apply(to_class)

print("Train rows:", len(train_df), "| Test rows:", len(test_df))

Class cut points: 47.4 68.77
Train rows: 9516 | Test rows: 3720


In [8]:
current_features = ["training_duration", "training_intensity", "training_load",
                    "heart_rate", "sleep_quality", "recovery_score", "stress_level"]

history_features = [c for c in df.columns
                    if c.startswith(("load_sum_", "load_mean_", "load_minus_mean_", "load_ratio_"))
                    or any(c == f"{col}_mean_{w}" for col in
                           ["training_intensity", "heart_rate", "sleep_quality",
                            "recovery_score", "stress_level"] for w in [7, 14])]

features = current_features + history_features

X_train, y_train = train_df[features], train_df["fatigue_class"]
X_test,  y_test  = test_df[features],  test_df["fatigue_class"]

print("Number of input features:", len(features))                  # should be 25
print("fatigue_index in features?", "fatigue_index" in features)   # must be False

Number of input features: 25
fatigue_index in features? False


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

xgb = Pipeline([
    ("fill_missing", SimpleImputer(strategy="median")),   # same fill as the other two models
    ("model", XGBClassifier(
        objective="multi:softprob",
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )),
])

t0 = time.perf_counter()
xgb.fit(X_train, y_train)
train_wall_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred  = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)
predict_wall_time = time.perf_counter() - t0

print(f"Training wall time:   {train_wall_time:.4f} seconds")
print(f"Prediction wall time: {predict_wall_time:.4f} seconds")

Training wall time:   3.3471 seconds
Prediction wall time: 0.0769 seconds


In [10]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    "roc_auc_ovr_macro": roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro"),
}
for k, v in metrics.items():
    print(f"{k:20s}: {v:.4f}")

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion matrix (rows = true, columns = predicted):")
print(pd.DataFrame(cm, index=[f"true_{n}" for n in class_names],
                   columns=[f"pred_{n}" for n in class_names]))

print("\n", classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

accuracy            : 0.6532
balanced_accuracy   : 0.6532
precision_macro     : 0.6535
recall_macro        : 0.6532
f1_macro            : 0.6534
f1_weighted         : 0.6535
roc_auc_ovr_macro   : 0.8376

Confusion matrix (rows = true, columns = predicted):
               pred_Low  pred_Moderate  pred_High
true_Low            922            330         52
true_Moderate       321            639        272
true_High            41            274        869

               precision    recall  f1-score   support

         Low       0.72      0.71      0.71      1304
    Moderate       0.51      0.52      0.52      1232
        High       0.73      0.73      0.73      1184

    accuracy                           0.65      3720
   macro avg       0.65      0.65      0.65      3720
weighted avg       0.65      0.65      0.65      3720



In [11]:
importances = pd.Series(xgb.named_steps["model"].feature_importances_, index=features)
importances = importances.sort_values(ascending=False)
print(importances.round(4))

training_load                 0.2610
load_ratio_14                 0.0800
load_minus_mean_14            0.0520
recovery_score                0.0382
training_duration             0.0317
load_sum_14                   0.0299
load_mean_14                  0.0297
load_ratio_7                  0.0291
load_mean_7                   0.0283
stress_level                  0.0282
recovery_score_mean_7         0.0276
sleep_quality                 0.0273
heart_rate_mean_7             0.0269
load_minus_mean_7             0.0269
recovery_score_mean_14        0.0266
sleep_quality_mean_14         0.0266
load_sum_7                    0.0266
sleep_quality_mean_7          0.0262
training_intensity_mean_7     0.0261
training_intensity            0.0259
heart_rate                    0.0256
heart_rate_mean_14            0.0256
stress_level_mean_14          0.0251
stress_level_mean_7           0.0247
training_intensity_mean_14    0.0245
dtype: float32


In [12]:
results = pd.DataFrame([{
    "model": "XGBoost",
    "train_rows": len(X_train),
    "test_rows": len(X_test),
    "n_features": len(features),
    "train_wall_time_sec": round(train_wall_time, 4),
    "predict_wall_time_sec": round(predict_wall_time, 4),
    "total_wall_time_sec": round(train_wall_time + predict_wall_time, 4),
    **{k: round(v, 4) for k, v in metrics.items()},
}])

per_class = pd.DataFrame(classification_report(
    y_test, y_pred, target_names=class_names, output_dict=True, zero_division=0)).T

results.to_csv("/kaggle/working/xgboost_metrics.csv", index=False)
per_class.to_csv("/kaggle/working/xgboost_per_class.csv")
importances.rename("importance").to_csv("/kaggle/working/xgboost_feature_importance.csv")

print(results.T)
print(os.listdir("/kaggle/working"))

                             0
model                  XGBoost
train_rows                9516
test_rows                 3720
n_features                  25
train_wall_time_sec     3.3471
predict_wall_time_sec   0.0769
total_wall_time_sec      3.424
accuracy                0.6532
balanced_accuracy       0.6532
precision_macro         0.6535
recall_macro            0.6532
f1_macro                0.6534
f1_weighted             0.6535
roc_auc_ovr_macro       0.8376
['xgboost_feature_importance.csv', 'xgboost_metrics.csv', '.virtual_documents', 'xgboost_per_class.csv']
